## Data Loading

In [ ]:
import pandas as pd

file1 = '/content/fhvhv_tripdata_2026-01.parquet'
file2 = '/content/fhvhv_tripdata_2026-02.parquet'

# Read the parquet files into pandas DataFrames
df1 = pd.read_parquet(file1)
df2 = pd.read_parquet(file2)

# Combine the two DataFrames into one table
combined_df = pd.concat([df1, df2], ignore_index=True)

# Display the first few rows of the combined DataFrame and its shape
print("Combined DataFrame head:")
print(combined_df.head())
print("\nCombined DataFrame shape:", combined_df.shape)

Combined DataFrame head:
  hvfhs_license_num dispatching_base_num originating_base_num  \
0            HV0003               B03404               B03404   
1            HV0003               B03404               B03404   
2            HV0003               B03404               B03404   
3            HV0005               B03406                 None   
4            HV0005               B03406                 None   

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:08:19 2026-01-01 00:09:08 2026-01-01 00:09:37   
4 2026-01-01 00:21:16 2026-01-01 00:26:30 2026-01-01 00:26:40   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  ...  \
0 2026-01-01 01:13:23           262            79       4.300  ...   
1 2026-01-01 00:22:42           195            52    

## Taxi Zone Integration

In [ ]:
taxi_zones_df = pd.read_csv('/content/taxi_zone_lookup.csv')

# Remove the 'service_zone' column as requested
taxi_zones_df = taxi_zones_df.drop(columns=['service_zone'])

# Display the head of the taxi zones DataFrame to understand its structure
print("Taxi Zones DataFrame head:")
print(taxi_zones_df.head())
print("\nTaxi Zones DataFrame shape:", taxi_zones_df.shape)

Taxi Zones DataFrame head:
   LocationID        Borough                     Zone
0           1            EWR           Newark Airport
1           2         Queens              Jamaica Bay
2           3          Bronx  Allerton/Pelham Gardens
3           4      Manhattan            Alphabet City
4           5  Staten Island            Arden Heights

Taxi Zones DataFrame shape: (265, 3)


In [ ]:
uber_trips_df = combined_df[combined_df['hvfhs_license_num'] == 'HV0003'].reset_index(drop=True)

# Merge for Pickup Location Information
uber_trips_df = uber_trips_df.merge(
    taxi_zones_df.rename(columns={'LocationID': 'PULocationID', 'Zone': 'PUZone', 'Borough': 'PUBorough'}),
    on='PULocationID',
    how='left'
)

# Merge for Dropoff Location Information
uber_trips_df = uber_trips_df.merge(
    taxi_zones_df.rename(columns={'LocationID': 'DOLocationID', 'Zone': 'DOZone', 'Borough': 'DOBorough'}),
    on='DOLocationID',
    how='left'
)

print("Uber Trips DataFrame after merging with taxi zone lookup:")
print(uber_trips_df.head())
print("\nNew DataFrame shape:", uber_trips_df.shape)

# Check for any missing values introduced by the merge (e.g., if some LocationIDs were not found)
print("\nMissing values after merge:")
print(uber_trips_df[['PUZone', 'PUBorough', 'DOZone', 'DOBorough']].isnull().sum())

Uber Trips DataFrame after merging with taxi zone lookup:
  hvfhs_license_num dispatching_base_num originating_base_num  \
0            HV0003               B03404               B03404   
1            HV0003               B03404               B03404   
2            HV0003               B03404               B03404   
3            HV0003               B03404               B03404   
4            HV0003               B03404               B03404   

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:29:54 2026-01-01 00:36:04 2026-01-01 00:36:32   
4 2026-01-01 00:07:33 2026-01-01 00:10:16 2026-01-01 00:11:14   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  ...  \
0 2026-01-01 01:13:23           262            79        4.30  ...   
1 2026-01-01 00:22:4

##Data Preperation

In [ ]:
# uber_trips_df = combined_df[combined_df['hvfhs_license_num'] == 'HV0003'].reset_index(drop=True)

pd.set_option('display.max_columns', None)
print("Uber Trips DataFrame head:")
print(uber_trips_df.head())
print(uber_trips_df.shape)
print("hvfhs_license_num: ",uber_trips_df['hvfhs_license_num'].unique())
print("dispatching_base_num: ", uber_trips_df['dispatching_base_num'].unique())
print("originating_base_num: ", uber_trips_df['originating_base_num'].unique())
print("originating_base_num value counts: \n", uber_trips_df['originating_base_num'].value_counts())
print("wav_request_flag: ", uber_trips_df['wav_request_flag'].unique())
print("wav_match_flag, : ", uber_trips_df['wav_match_flag'].unique())
print("congestion_surcharge value counts: \n", uber_trips_df['congestion_surcharge'].value_counts())
print("airport_fee value counts: \n", uber_trips_df['airport_fee'].value_counts())
print("shared_request_flag value counts: \n", uber_trips_df['shared_request_flag'].value_counts())
print("shared_match_flag value counts: \n", uber_trips_df['shared_match_flag'].value_counts())
print("access_a_ride_flag value counts: \n", uber_trips_df['access_a_ride_flag'].value_counts())
print("cbd_congestion_fee value counts: \n", uber_trips_df['cbd_congestion_fee'].value_counts())


Uber Trips DataFrame head:
  hvfhs_license_num dispatching_base_num originating_base_num  \
0            HV0003               B03404               B03404   
1            HV0003               B03404               B03404   
2            HV0003               B03404               B03404   
3            HV0003               B03404               B03404   
4            HV0003               B03404               B03404   

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:29:54 2026-01-01 00:36:04 2026-01-01 00:36:32   
4 2026-01-01 00:07:33 2026-01-01 00:10:16 2026-01-01 00:11:14   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  \
0 2026-01-01 01:13:23           262            79        4.30       1133   
1 2026-01-01 00:22:42           195    

In [ ]:
pu_location_ids_to_remove = [264, 265, 1, 132, 138] #Remove any Unknown, N/A, EWR, JFK, and LGA pickups respectively to avoid anomolies in waitimes
do_location_ids_to_remove = [264, 265, 1] #Remove any Unknown, N/A, and EWR dropoffs respectively to avoid anomolies

print("\nCounts for PULocationID to be removed:")
print(uber_trips_df[uber_trips_df['PULocationID'].isin(pu_location_ids_to_remove)]['PULocationID'].value_counts().sort_index())

print("\nCounts for DOLocationID to be removed:")
print(uber_trips_df[uber_trips_df['DOLocationID'].isin(do_location_ids_to_remove)]['DOLocationID'].value_counts().sort_index())


uber_trips_df = uber_trips_df[
    (uber_trips_df['originating_base_num'] == 'B03404') &
    (uber_trips_df['wav_request_flag'] == 'N') &
    (uber_trips_df['wav_match_flag'] == 'N') &
    (uber_trips_df['access_a_ride_flag'] == 'N') &
    (uber_trips_df['shared_match_flag'] == 'N') &
    (uber_trips_df['shared_request_flag'] == 'N') &
    (~uber_trips_df['PULocationID'].isin(pu_location_ids_to_remove)) &
    (~uber_trips_df['DOLocationID'].isin(do_location_ids_to_remove))
].reset_index(drop=True)

print("Uber Trips DataFrame after filtering:")
print(uber_trips_df.head())
print("New DataFrame shape:", uber_trips_df.shape)

print("\nUnique counts for each column:")
for column in uber_trips_df.columns:
    print(f"{column}: {uber_trips_df[column].nunique()}")


Counts for PULocationID to be removed:
PULocationID
1           3
132    466442
138    505133
265      1391
Name: count, dtype: int64

Counts for DOLocationID to be removed:
DOLocationID
1       197837
264        743
265    1191310
Name: count, dtype: int64
Uber Trips DataFrame after filtering:
  hvfhs_license_num dispatching_base_num originating_base_num  \
0            HV0003               B03404               B03404   
1            HV0003               B03404               B03404   
2            HV0003               B03404               B03404   
3            HV0003               B03404               B03404   
4            HV0003               B03404               B03404   

     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:29:54 2026-01-01 00:36:04 2026-01-01

In [ ]:
columns_to_drop = [
    'hvfhs_license_num',
    'dispatching_base_num',
    'originating_base_num',
    'base_passenger_fare',
    'bcf',
    'sales_tax',
    'congestion_surcharge',
    'airport_fee',
    'shared_request_flag',
    'shared_match_flag',
    'access_a_ride_flag',
    'wav_request_flag',
    'wav_match_flag',
    'cbd_congestion_fee'
]

# Drop columns that have only one unique value or are otherwise deemed unnecessary
uber_trips_df = uber_trips_df.drop(columns=columns_to_drop)

print("Uber Trips DataFrame after dropping columns:")
print(uber_trips_df.head())
print("New DataFrame shape:", uber_trips_df.shape)

Uber Trips DataFrame after dropping columns:
     request_datetime   on_scene_datetime     pickup_datetime  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31 2026-01-01 00:54:30   
1 2026-01-01 00:09:12 2026-01-01 00:12:17 2026-01-01 00:12:39   
2 2026-01-01 00:16:16 2026-01-01 00:29:33 2026-01-01 00:31:34   
3 2026-01-01 00:29:54 2026-01-01 00:36:04 2026-01-01 00:36:32   
4 2026-01-01 00:07:33 2026-01-01 00:10:16 2026-01-01 00:11:14   

     dropoff_datetime  PULocationID  DOLocationID  trip_miles  trip_time  \
0 2026-01-01 01:13:23           262            79        4.30       1133   
1 2026-01-01 00:22:42           195            52        1.89        604   
2 2026-01-01 00:55:21            25           181        1.84       1427   
3 2026-01-01 01:02:28           162           113        2.84       1556   
4 2026-01-01 00:14:06            22            22        0.66        172   

   tolls  tips  driver_pay  PUBorough            PUZone  DOBorough  \
0    0.0  0.00       21.10  Manhattan

In [ ]:
print("Missing values before date conversion:")
print(uber_trips_df.isnull().sum())

# Convert datetime columns to datetime objects
datetime_cols = ['request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime']
for col in datetime_cols:
    uber_trips_df[col] = pd.to_datetime(uber_trips_df[col], errors='coerce')

print("\nMissing values after date conversion (if any coercion occurred):")
print(uber_trips_df.isnull().sum())

print("\nData types after conversion:")
print(uber_trips_df[datetime_cols].dtypes)

print("\n", uber_trips_df.head())
print("\n", uber_trips_df.describe())


Missing values before date conversion:
request_datetime     0
on_scene_datetime    0
pickup_datetime      0
dropoff_datetime     0
PULocationID         0
DOLocationID         0
trip_miles           0
trip_time            0
tolls                0
tips                 0
driver_pay           0
PUBorough            0
PUZone               0
DOBorough            0
DOZone               0
dtype: int64

Missing values after date conversion (if any coercion occurred):
request_datetime     0
on_scene_datetime    0
pickup_datetime      0
dropoff_datetime     0
PULocationID         0
DOLocationID         0
trip_miles           0
trip_time            0
tolls                0
tips                 0
driver_pay           0
PUBorough            0
PUZone               0
DOBorough            0
DOZone               0
dtype: int64

Data types after conversion:
request_datetime     datetime64[us]
on_scene_datetime    datetime64[us]
pickup_datetime      datetime64[us]
dropoff_datetime     datetime64[us]
dtype

## Feature Engineering

In [ ]:
uber_trips_df['time_from_request_to_pickup'] = (uber_trips_df['on_scene_datetime'] - uber_trips_df['request_datetime']).dt.total_seconds()
uber_trips_df['day_of_week'] = uber_trips_df['request_datetime'].dt.day_name()
uber_trips_df['hour_bucket'] = uber_trips_df['request_datetime'].dt.hour

print("Uber Trips DataFrame with new features:")
print(uber_trips_df[['request_datetime', 'on_scene_datetime', 'time_from_request_to_pickup', 'day_of_week', 'hour_bucket']].head())
print("\nNew DataFrame shape:", uber_trips_df.shape)

print("\nMissing values after feature engineering:")
print(uber_trips_df.isnull().sum())

Uber Trips DataFrame with new features:
     request_datetime   on_scene_datetime  time_from_request_to_pickup  \
0 2026-01-01 00:50:37 2026-01-01 00:52:31                        114.0   
1 2026-01-01 00:09:12 2026-01-01 00:12:17                        185.0   
2 2026-01-01 00:16:16 2026-01-01 00:29:33                        797.0   
3 2026-01-01 00:29:54 2026-01-01 00:36:04                        370.0   
4 2026-01-01 00:07:33 2026-01-01 00:10:16                        163.0   

  day_of_week  hour_bucket  
0    Thursday            0  
1    Thursday            0  
2    Thursday            0  
3    Thursday            0  
4    Thursday            0  

New DataFrame shape: (23474097, 18)

Missing values after feature engineering:
request_datetime               0
on_scene_datetime              0
pickup_datetime                0
dropoff_datetime               0
PULocationID                   0
DOLocationID                   0
trip_miles                     0
trip_time                     

## Export To Parquet

In [ ]:
uber_trips_df.to_parquet('uber_trips.parquet', index=False)
print("DataFrame successfully exported to uber_trips.parquet")

DataFrame successfully exported to uber_trips.parquet
